# The Travelling *Swiftie* Problem

**An open Travelling Salesman Problem inspired by Taylor Swift's *London Boy*.**

This notebook contains the Python implementation accompanying [*The Travelling Swiftie Problem*](https://doi.org/10.1080/2058802X.2024.2388417), published in the OR Society's *Impact* magazine.

The aim is to find the shortest route visiting nine London locations once, **finishing in Brixton**. The notebook concentrates on the implementation and checks; the article provides the broader story, interpretation and applications of routing optimisation.

We use the original distance matrix supplied with the repository and an **exact, brute-force search**.

## 1. Problem and assumptions

The conventional Travelling Salesman Problem (TSP) seeks a shortest route that visits every location before returning to its starting point. Here, we solve an **open TSP**: no return trip is required, the starting point is free to vary, and **Brixton is fixed as the final destination**.

Following the assumptions described in the article:

- Each of the nine locations is visited exactly once; repeated mentions in the song do not create additional visits.
- The couple travel by **scooter**, with suitable parking assumed available.
- We minimise **distance travelled (miles)**, not journey time; traffic effects are ignored.

There are eight locations to order before Brixton, giving **8! = 40,320** candidate routes. This is small enough to search exhaustively.

## 2. Libraries and distance data

The matrix below contains pairwise distances (in miles), copied **unchanged** from the original notebook. The values are rounded to one decimal place. The exact collection method for these particular matrix entries is **not documented** in the original code or accompanying article, so no data source is asserted here.

The final entry, `Dummy End`, is a modelling device from the original implementation. Travel from Brixton to it costs zero, whereas travel from any other real stop to it has infinite cost. It is **not** an additional destination on the London tour.

In [1]:
from itertools import permutations
from math import factorial, isclose

import numpy as np

In [2]:
stops = ["Camden Market", "Highgate", "West End", "Shoreditch", "Hackney", "Bond Street", "SoHo", "Hampstead Heath", "Brixton", "Dummy End"]

distance_matrix = np.array([
    [0, 2.1, 2.6, 4.2, 5, 3, 2.5, 1.7, 6.3, np.inf],
    [2.1, 0, 4.5, 4.7, 4.8, 4.7, 4.3, 2.2, 8.1, np.inf],
    [2.6, 4.5, 0, 3.4, 5.2, 0.9, 0.8, 4.8, 4.4, np.inf],
    [4.2, 4.7, 3.4, 0, 2.3, 3.8, 3.2, 5.5, 6.2, np.inf],
    [5, 4.8, 5.2, 2.3, 0, 5.7, 5.3, 5.9, 8.2, np.inf],
    [3, 4.7, 0.9, 3.8, 5.7, 0, 0.3, 4.6, 4.8, np.inf],
    [2.5, 4.3, 0.8, 3.2, 5.3, 0.3, 0, 4.2, 5.5, np.inf],
    [1.7, 2.2, 4.8, 5.5, 5.9, 4.6, 4.2, 0, 9.4, np.inf],
    [6.3, 8.1, 4.4, 6.2, 8.2, 4.8, 5.5, 9.4, 0, 0],
    [np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, 0]
], dtype=float)

print(f"Real locations: {len(stops) - 1}")
print(f"Distance matrix shape: {distance_matrix.shape}")
print(f"Possible routes with Brixton fixed last: {factorial(len(stops) - 2):,}")

Real locations: 9
Distance matrix shape: (10, 10)
Possible routes with Brixton fixed last: 40,320


## 3. Exact solution by brute-force search

For each candidate, we sum the distances between successive stops. The dummy leg adds zero miles, so it does not change the objective value. We keep track of all equally optimal routes, allowing for small rounding differences.

The function **returns** results rather than printing them, making it easier to check or reuse.

In [6]:
def travelling_swiftie_problem(d_mat, stops):
    n = len(d_mat)
    best_cost = [np.inf]
    best_path = []
    best_distances = []

    # Fix Brixton as the final destination and exclude the dummy node
    brixton = stops.index("Brixton")
    dummy = stops.index("Dummy End")
    remaining = [i for i in range(n) if i not in (brixton, dummy)]

    for permutation in permutations(remaining):
        path = list(permutation) + [brixton]
        distances = [d_mat[path[i]][path[i+1]] for i in range(len(path)-1)]
        cost = sum(distances)

        if not np.isfinite(cost):
            continue

        if cost < best_cost[0]:
            best_cost = [cost]
            best_path = [path]
            best_distances = [distances]
        elif np.isclose(cost, best_cost[0], rtol=0, atol=1e-9):
            best_cost.append(cost)
            best_path.append(path)
            best_distances.append(distances)

    ordered_stops = [[stops[i] for i in path] for path in best_path]

    print(f"The shortest distance travelled is: {best_cost} miles.")
    print(f"The path taken is: {ordered_stops}")
    print(f"The distances travelled between stops are: {best_distances}")

    return best_cost[0]

In [7]:
optimal_distance = travelling_swiftie_problem(distance_matrix, stops)

The shortest distance travelled is: [19.0] miles.
The path taken is: [['Hackney', 'Shoreditch', 'Highgate', 'Hampstead Heath', 'Camden Market', 'SoHo', 'Bond Street', 'West End', 'Brixton']]
The distances travelled between stops are: [[2.3, 4.7, 2.2, 1.7, 2.5, 0.3, 0.9, 4.4]]


## 4. Comparison with the song-order route

For context, the article also compares the optimised route against the order in which the locations are **first mentioned** in the song. We can calculate the distance of that order using the matrix stored in this notebook.

In [8]:
song_order = [
    "Camden Market", "Highgate", "West End", "Shoreditch",
    "Hackney", "Bond Street", "Hampstead Heath", "SoHo", "Brixton"
]

# Calculate the total distance of the song-order route
song_indices = [stops.index(name) for name in song_order]

song_distance = sum(
    distance_matrix[a][b]
    for a, b in zip(song_indices, song_indices[1:])
)

# Compare the two routes
print(f"Song-order route: {song_distance:.1f} miles")
print(f"Optimal route: {optimal_distance:.1f} miles")
print(f"Distance saved: {song_distance - optimal_distance:.1f} miles")

Song-order route: 32.3 miles
Optimal route: 19.0 miles
Distance saved: 13.3 miles
